# Trace: AI meeting assistant, on a free Colab GPU

Upload a meeting recording and get the **raw transcript, refined transcript, minutes, decisions and action items**,
all grounded in what was actually said.

**How to use**
1. **Runtime → Change runtime type → T4 GPU** (free), then **Save**.
2. **Runtime → Run all.** Setup takes about 10-15 minutes the first time (installing and downloading the models).
3. The last cell prints a link like `https://….trycloudflare.com`. Open it and upload a recording.

Keep this tab open while you use the app; closing it or a Colab timeout stops the server.

Optional: for the most accurate speaker labels, add a Hugging Face token below (accept the terms of
`pyannote/speaker-diarization-community-1` and `pyannote/segmentation-3.0` on huggingface.co first).
Without a token the app uses ECAPA voice embeddings, which need no account.

In [ ]:
#@title 1. Settings { display-mode: "form" }
REPO_URL = "https://github.com/shivanshi-tyagi30/ML-Bootcamp-interIIT.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
LM1_MODEL = "qwen3:8b"  #@param ["qwen3:8b", "qwen3:14b", "qwen3:4b"]
LM2_MODEL = "gemma3:12b"  #@param ["gemma3:12b", "gemma3:4b"]
HF_TOKEN = ""  #@param {type:"string"}
NUM_SPEAKERS = 0  #@param {type:"integer"}
print("Settings saved. 0 speakers = detect automatically.")

In [ ]:
#@title 2. Check the GPU { display-mode: "form" }
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if out.returncode != 0:
    raise SystemExit("No GPU found. Go to Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.")
print("GPU:", out.stdout.strip())

In [ ]:
#@title 3. Get the code { display-mode: "form" }
import os, shutil
os.chdir("/content")
if os.path.exists("/content/app"):
    shutil.rmtree("/content/app")
!git clone -q --depth 1 -b {BRANCH} {REPO_URL} /content/app
print("Code downloaded.")

In [ ]:
#@title 4. Install Python and system packages (~3 min) { display-mode: "form" }
!apt-get -qq update > /dev/null && apt-get -qq install -y libmagic1 ffmpeg zstd > /dev/null
!pip -q install -r /content/app/backend/requirements-colab.txt
if HF_TOKEN.strip():
    !pip -q install pyannote.audio
# faster-whisper needs the CUDA libraries that come with Colab's PyTorch wheels on its library path.
import glob, os
libs = sorted({os.path.dirname(p) for p in glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/*/lib/*.so*")})
os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("Installed.")

In [ ]:
#@title 5. Install and start Ollama (runs the two language models) { display-mode: "form" }
import subprocess, time, urllib.request
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                          env={**os.environ, "OLLAMA_KEEP_ALIVE": "-1m"})
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("Ollama did not start; see /content/ollama.log")
print("Ollama is running.")

In [ ]:
#@title 6. Build the website (~2 min) { display-mode: "form" }
NODE = "v22.11.0"
!curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C /content
os.environ["PATH"] = f"/content/node-{NODE}-linux-x64/bin:" + os.environ["PATH"]
!cd /content/app/frontend && npm ci --silent --no-audit --no-fund && npm run build --silent
print("Website built.")

In [ ]:
#@title 7. Configure and download all models (~5-10 min the first time) { display-mode: "form" }
import re
os.chdir("/content/app/backend")
env = open(".env.colab").read()
for key, value in {"LM1_MODEL": LM1_MODEL, "LM2_MODEL": LM2_MODEL, "HF_TOKEN": HF_TOKEN.strip(),
                   "DIARIZATION_NUM_SPEAKERS": str(NUM_SPEAKERS)}.items():
    env = re.sub(rf"^{key}=.*$", f"{key}={value}", env, flags=re.M)
open(".env", "w").write(env)
!python -m app.prefetch

In [ ]:
#@title 8. Start the app and get your public link { display-mode: "form" }
import json, re, subprocess, time, urllib.request
os.chdir("/content/app/backend")
backend = subprocess.Popen(["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
                           stdout=open("/content/backend.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
print("Loading every model into the GPU before opening the site...")
health = None
for _ in range(900):
    try:
        health = json.load(urllib.request.urlopen("http://localhost:8000/api/health", timeout=3))
        break
    except Exception:
        if backend.poll() is not None:
            raise SystemExit("The backend stopped; see /content/backend.log:\n" + open("/content/backend.log").read()[-3000:])
        time.sleep(1)
if not health or not health.get("ready"):
    print("Started, but not everything loaded:", (health or {}).get("not_loaded"), "- see /content/backend.log")
else:
    print("All models loaded.")

!curl -fsSL -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break
    time.sleep(1)
if not url:
    raise SystemExit("Could not open the public link; see /content/tunnel.log")
from IPython.display import HTML, display
display(HTML(f'<h2>Trace is ready: <a href="{url}" target="_blank">{url}</a></h2>'
             '<p>Keep this tab open while you use it.</p>'))

In [ ]:
#@title (Optional) Show the last lines of the server log { display-mode: "form" }
print(open("/content/backend.log").read()[-4000:])